In [ ]:
!pip install datasets
from datasets import load_dataset
from IPython.display import clear_output
clear_output()

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
ds = load_dataset('Z-Jafari/PersianQuAD_Factoid_Test')

In [ ]:
ds

In [ ]:
ds['train'][0]['answers']

In [ ]:
i = 0
for d in ds['train']:
  if len(d['answers']['text'][0]) > 150:
    if 'چرا' in d['question']:
      print(d['question'], ' ', d['answers']['text'][0])
      print('--')
      i += 1

print(i)

In [ ]:

i = 0
for d in ds['train']:
  if (d['is_factoid'] == 'False') and (len(d['answers']['text'][0]) > 150):
    print(d['question'], ' ', d['answers']['text'][0])
    print('--')
    i += 1

print(i)

In [ ]:
i = 0
for d in ds['test']:
  #if d['is_factoid'] == 'False':
    #print(d['question'], '\n', d['answers']['text'][0])
    for a in d['answers']['text']:
      if len(a)>60:
        i += 1
        #print(a, ': ', len(a))
print(i)

In [ ]:
i = 0
for d in ds['train']:
  if (d['is_factoid'] == 'False') and (len(d['answers']['text'][0]) > 20):
    print(d['answers']['text'][0])
    i += 1
print(i)

In [ ]:
for d in ds['test']:
  print(d['question'])
  for a in d['answers']['text']:
    print(a)
  print('***')

In [ ]:
squad = load_dataset('squad')

In [ ]:
squad

In [ ]:
# تقسیم بخش train به 90% آموزش و 10% اعتبارسنجی به صورت متناسب
split_dict = ds['train'].train_test_split(
    test_size=0.1,
    seed=42,
    )

# جایگزینی در متغیر ds
ds['train'] = split_dict['train']
ds['validation'] = split_dict['test']

In [ ]:
ds

In [ ]:
i = 0
for d in ds['train']:
  print(not(d['is_factoid']))
  if i > 3:
    break
  if d['is_factoid']:
    i += 1
print(i)
j = 0
for d in ds['test']:
  if d['is_factoid']:
    j += 1
print(j)

#print(i/j)

In [ ]:
# Define the filter function
def filter_conditions(example):
    # 1. Check if 'is_factoid' is the string 'True' (or boolean True, just in case)
    is_factoid = str(example['is_factoid']) == 'True'

    # 2. Check if there is at least one answer, and its length is less than 20
    has_short_answer = (
        example['answers'] is not None
        and len(example['answers']['text']) > 0
        and len(example['answers']['text'][0]) < 20
    )

    return is_factoid and has_short_answer

# Apply the filter to the entire DatasetDict (train and test splits)
filtered_ds = ds.filter(filter_conditions)

# Print to verify the new sizes
print(filtered_ds)

In [ ]:
# Define the filter function
ANS_LEN = 60
def filter_Len_conditions(example):
    has_short_answer = (
        example['answers'] is not None
        and len(example['answers']['text'][0]) < ANS_LEN
    )
    return has_short_answer

# Apply the filter to the entire DatasetDict (train and test splits)
filtered_len_ds = ds.filter(filter_Len_conditions)

# Print to verify the new sizes
print(filtered_len_ds)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# 1. Convert your Hugging Face DatasetDict to Pandas DataFrames
# Replace `my_dataset_dict` with the actual name of your DatasetDict variable
df_train = filtered_len_ds['train'].to_pandas()
df_test = filtered_len_ds['test'].to_pandas()

# Add a split column so we can track train vs test distributions
df_train['split'] = 'train'
df_test['split'] = 'test'

# Combine them for a holistic analysis
df = pd.concat([df_train, df_test], ignore_index=True)

# 2. Function to safely extract the character length of the first gold answer
def extract_first_answer_len(answers_field):
    if isinstance(answers_field, dict) and 'text' in answers_field:
        text_list = answers_field['text']
        if len(text_list) > 0:
            # Takes the primary answer string and measures character length
            return len(str(text_list[0]))
    return 0

# Apply the function and create the 18-character threshold group
df['answer_length'] = df['answers'].apply(extract_first_answer_len)
df['length_group'] = df['answer_length'].apply(lambda x: '< 18 chars' if x < 18 else '>= 18 chars')

# 3. ANALYSIS 1: How question types align with length boundaries (Entire Dataset)
print("=" * 50)
print(" ANALYSIS 1: QUESTION TYPE VS. ANSWER LENGTH ")
print("=" * 50)
contingency_table = pd.crosstab(df['is_factoid'], df['length_group'])
proportions_table = pd.crosstab(df['is_factoid'], df['length_group'], normalize='index') * 100

print("\n--- Raw Counts ---")
print(contingency_table)
print("\n--- Proportions by Question Type (%) ---")
print(proportions_table.round(2))

# Statistical Significance Test
chi2, p_value, _, _ = chi2_contingency(contingency_table)
print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"p-value: {p_value:.4e} (If < 0.05, length and type are strictly linked)")


# 4. ANALYSIS 2: The Distribution Mismatch (Train vs. Test Split)
print("\n" + "=" * 50)
print(" ANALYSIS 2: SPLIT MISMATCH DETECTED (TRAIN VS. TEST) ")
print("=" * 50)
split_type_table = pd.crosstab(df['split'], df['is_factoid'], normalize='index') * 100
print("\n--- Question Type Proportions across Splits (%) ---")
print(split_type_table.round(2))


# 5. Generate a Publication-Ready Plot
plt.figure(figsize=(10, 6))
# Using a split-based boxplot to show the difference visually
sns.boxplot(
    data=df,
    x='is_factoid',
    y='answer_length',
    hue='split',
    palette='muted',
    log_scale=True
)
plt.axhline(y=18, color='crimson', linestyle='--', linewidth=2, label='18-Char Threshold')
plt.title('Structural Comparison: Answer Lengths by Split and Question Type', fontsize=12, pad=15)
plt.xlabel('Is Factoid? (DeepSeek Silver Labels)', fontsize=10)
plt.ylabel('Answer Length (Characters - Log Scale)', fontsize=10)
plt.legend(title='Dataset Split')
plt.tight_layout()

# Save the visualization directly for your manuscript
plt.savefig('persianquad_structural_mismatch.png', dpi=300)
plt.show()